### Pre-Processing (Idea Specific)

In [13]:
from shared.config import PROCESSED
import pandas as pd
import sys
sys.path.insert(0, '..')


CLEAN = PROCESSED / 'clean'
plays = pd.read_parquet(CLEAN / 'player_play.parquet')
games = pd.read_parquet(PROCESSED / 'parquet' /
                        'nfl-big-data-bowl-2027__games.parquet')
players = pd.read_parquet(PROCESSED / 'parquet' /
                          'nfl-big-data-bowl-2027__players.parquet')

df = (plays.merge(games[['game_id', 'season', 'season_type']], on='game_id')
           .merge(players[['nfl_id', 'nfl_position']], on='nfl_id'))

steps = {}
steps['all player-plays'] = len(df)
df = df[df['nfl_position'].isin(['T', 'G', 'C'])]
steps['OL by nfl_position'] = len(df)
df = df[df['pass_rushers_encountered'].notna() & df['route_ran'].isna()]
steps['pass blocking'] = len(df)
df = df[df['season_type'].eq('REG')]
steps['regular season only'] = len(df)
df = df[df['play_nullified_by_penalty'].ne('Y')]
steps['no nullified plays'] = len(df)
# df = df[df['season'].ne(2025)]
# steps['2025 held out'] = len(df)

df['short_dropback'] = df['dropback_duration'] <= 2
pd.Series(steps)

all player-plays       314197
OL by nfl_position     107720
pass blocking           60175
regular season only     48689
no nullified plays      48688
dtype: int64

## Verify Short Dropback Ratio

In [14]:
df['short_dropback'].value_counts()

short_dropback
False    40077
True      8611
Name: count, dtype: int64

## Head to Validate Before Continuing

In [15]:
df[['nfl_id', 'game_id', 'play_id', 'season', 'nfl_position',
    'lined_up_position', 'dropback_duration', 'pressure_allowed']].head(100)

,nfl_id,game_id,play_id,season,nfl_position,lined_up_position,dropback_duration,pressure_allowed
0,57183,2025111610,2279,2025,T,T,2.079,False
8,55976,2023091801,56,2023,T,T,1.835,False
10,57183,2025122701,63,2025,T,T,2.887,False
16,55973,2023101503,3590,2023,G,G,1.635,False
21,57238,2026010405,2568,2025,C,C,2.665,False
...,...,...,...,...,...,...,...,...
643,57330,2025122111,626,2025,G,G,2.975,False
647,55870,2024091510,2211,2024,T,T,1.343,False
648,55874,2024111700,3469,2024,T,T,4.776,False
667,58359,2025092113,3981,2025,T,T,4.097,True


## Pressure Allowed Shared

In [16]:
df['pressure_allowed'].astype(float).mean(
), df['pressure_allowed'].isna().mean()

(np.float64(0.08425073940190601), np.float64(0.0))

## Per Player Counts

In [17]:
per_player = df.groupby('nfl_id').size()
per_player.describe(), (per_player >= 100).sum()

(count      89.000000
 mean      547.056180
 std       514.171871
 min         1.000000
 25%        98.000000
 50%       433.000000
 75%       787.000000
 max      1818.000000
 dtype: float64,
 np.int64(66))

In [18]:
per_player.head(10)

len(per_player)

89

### Step 2: Pre and Post Snap Windowing

In [20]:
KEYS = ['game_id', 'play_id', 'nfl_id']
COLS = KEYS + ['time', 'x', 'y', 's', 's_sg', 'a_sg', 'event', 'glitch_track']

wanted = df[KEYS]
parts = []
for yr in (2023, 2024, 2025):
    g = pd.read_parquet(CLEAN / f'game_tracking_{yr}.parquet', columns=COLS)
    parts.append(g.merge(wanted, on=KEYS))
trk = pd.concat(parts, ignore_index=True)
del parts, g

trk.shape, trk[KEYS].drop_duplicates().shape[0]

((3671683, 11), 48688)

In [21]:
len(df), len(df) - 48688

(48688, 0)

### Step 3. Cut Window

In [ ]:
trk = trk[~trk['glitch_track']]

snap = (trk[trk['event'].eq('ball_snap')]
        .groupby(['game_id', 'play_id'])['time'].min()
        .rename('snap_time'))
trk = trk.merge(snap, on=['game_id', 'play_id'])

trk['t'] = (pd.to_datetime(trk['time']) -
            pd.to_datetime(trk['snap_time'])).dt.total_seconds().round(1)
win = trk[trk['t'].between(-0.3, 1.5)]

frames = win.groupby(KEYS)['t'].agg(['min', 'max', 'size'])
full = frames[(frames['min'] <= -0.3) & (frames['max'] >= 1.5)]
win = win.merge(full.index.to_frame(index=False), on=KEYS)

len(frames), len(full), full['size'].value_counts().head()

(48649,
 48607,
 size
 19    48607
 Name: count, dtype: int64)

In [23]:
win.to_parquet(CLEAN / 'ol_passset_window.parquet', index=False)

## Step 5. Feature Selection

In [24]:
cr = pd.read_parquet(PROCESSED / 'parquet' / 'nfl-big-data-bowl-2027__combine_results.parquet',
                     columns=['nfl_id', 'combine_weight'])

stance = (win[win['t'] < 0].groupby(KEYS)[['x', 'y']].mean()
          .rename(columns={'x': 'x0', 'y': 'y0'}))
end = (win[win['t'] == 1.5].set_index(KEYS)[['x']]
       .rename(columns={'x': 'x_end'}))
first_step = (win[win['t'].between(0, 0.5)].groupby(KEYS)['a_sg'].max()
              .rename('peak_accel_05'))
peak_speed = (win[win['t'].between(0, 1.5)].groupby(KEYS)['s_sg'].max()
              .rename('peak_speed'))

feat = pd.concat([stance, end, first_step, peak_speed], axis=1).reset_index()
feat['retreat_depth'] = feat['x0'] - feat['x_end']   # positive = moved back

feat = feat.merge(cr, on='nfl_id', how='left')
feat['mass_kg'] = feat['combine_weight'] * 0.4536
feat['momentum'] = feat['mass_kg'] * feat['peak_speed'] * 0.9144   # kg*m/s

feat[['retreat_depth', 'peak_accel_05', 'momentum']].describe().round(2)

,retreat_depth,peak_accel_05,momentum
count,48607.00,48607.00,48607.00
mean,1.79,2.32,326.51
std,1.20,0.80,98.03
min,-4.11,-1.78,15.45
25%,1.10,1.76,257.32
50%,1.93,2.32,321.69
75%,2.62,2.88,392.12
max,6.00,7.44,753.31


In [ ]:
feat[['retreat_depth', 'peak_accel_05', 'peak_speed',
      'momentum', 'combine_weight']].corr().round(2)

,retreat_depth,peak_accel_05,peak_speed,momentum,combine_weight
retreat_depth,1.00,0.29,0.37,0.37,0.04
peak_accel_05,0.29,1.00,0.65,0.66,0.10
peak_speed,0.37,0.65,1.00,0.99,0.06
momentum,0.37,0.66,0.99,1.00,0.19
combine_weight,0.04,0.10,0.06,0.19,1.00


Play momentum and speed are highly correlated, we need to determine if at the player level this matters.

In [ ]:
pl = feat.groupby('nfl_id')[
    ['peak_speed', 'momentum', 'combine_weight']].mean()
pl.corr().round(2), len(pl)

(                peak_speed  momentum  combine_weight
 peak_speed            1.00      0.95            0.18
 momentum              0.95      1.00            0.48
 combine_weight        0.18      0.48            1.00,
 89)

Since they are still correlated, include them both in the modeling and leave it to the algo to decide.

In [ ]:
feat = feat[KEYS + ['retreat_depth',
                    'peak_accel_05', 'peak_speed', 'combine_weight']]
feat = feat.merge(df[KEYS + ['season', 'nfl_position', 'lined_up_position', 'short_dropback',
                             'dropback_duration', 'pass_rushers_encountered', 'pressure_allowed']],
                  on=KEYS, how='left')
feat.to_parquet(CLEAN / 'ol_passset_features.parquet', index=False)
feat.shape, feat.isna().sum().sum()

((48607, 14), np.int64(0))